section 1 - imports & utilities:

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import trange

section 2 - dataset: 

XOR and 3-bit parity tasks

In [2]:
# XOR
X_xor = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y_xor = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

# 3-bit parity
from itertools import product
X_parity = torch.tensor(list(product([0,1], repeat=3)), dtype=torch.float32)
y_parity = X_parity.sum(dim=1) % 2
y_parity = y_parity.unsqueeze(1).float()

section 3 - model definitions:

(A) Simple perceptron:

In [3]:
class Perceptron(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, 1)
    def forward(self, x):
        return torch.sigmoid(self.fc(x))

(B) Two-layer MLP (backprop part):

In [4]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

(C) Dendritic Neuron (prototype):

this will later be replaced with a custom local learning model. see how to incorporate dendritic principles into this. this is where background research comes into play.

In [ ]:
# minimal_dll.py
import torch
import torch.nn.functional as F

# ---- activations ----
def tanh(x):   return torch.tanh(x)
def dtanh(x):  return 1.0 - torch.tanh(x)**2
def sigmoid(x): return torch.sigmoid(x)
def dsigmoid(x): s = torch.sigmoid(x); return s * (1 - s)

class DendriticDLL(torch.nn.Module):
    """
    Single-neuron with M dendritic subunits:
      h = f(W x)         # dendritic nonlinearity
      y = g(v^T h)       # soma nonlinearity (binary) OR logits = V h (multi-class)
    DLL credit assignment via learned feedback theta_y.
    """
    def __init__(self, in_dim, n_dendrites=2, out_dim=1,
                 f=tanh, df=dtanh, g=sigmoid, dg=dsigmoid,
                 lr=1e-2, theta_lr=1e-2, clamp=50.0, multi_class=False, device="cpu"):
        super().__init__()
        self.in_dim = in_dim
        self.M = n_dendrites
        self.out_dim = out_dim
        self.multi_class = multi_class
        self.f, self.df = f, df
        self.g, self.dg = g, dg
        self.lr = lr
        self.theta_lr = theta_lr
        self.clamp = clamp
        self.device = torch.device(device)

        # weights
        self.W = torch.nn.Parameter(torch.empty(self.M, self.in_dim).normal_(0, 0.05))
        if self.multi_class:
            self.V = torch.nn.Parameter(torch.empty(self.out_dim, self.M).normal_(0, 0.05))
        else:
            self.v = torch.nn.Parameter(torch.empty(self.M).normal_(0, 0.05))

        # learned feedback (DLL)
        # maps output-layer local errors back to dendrites
        # shape: (M, out_dim)
        self.theta_y = torch.nn.Parameter(torch.empty(self.M, self.out_dim).normal_(0, 0.05))

    def forward(self, x):
        """
        x: [B, in_dim]
        returns:
          if multi_class: logits [B, out_dim]
          else: y in (0,1) [B, 1]
        """
        self.x = x
        self.a_h = x @ self.W.t()         # [B, M]
        self.h = self.f(self.a_h)         # [B, M]
        if self.multi_class:
            self.logits = self.h @ self.V.t()   # [B, C]
            return self.logits
        else:
            self.a_y = self.h @ self.v     # [B]
            self.y = self.g(self.a_y)      # [B]
            return self.y.unsqueeze(-1)

    @torch.no_grad()
    def dll_step(self, target):
        """
        One DLL update step using current forward caches.
        target:
          - binary: [B, 1] in {0,1}
          - multi-class: [B] class indices or one-hot [B, C]
        """
        B = self.x.shape[0]

        # ----- output error (local) -----
        if self.multi_class:
            # targets: class indices -> CE with logits
            if target.dim() == 2:  # one-hot -> indices
                targ_idx = target.argmax(dim=1)
            else:
                targ_idx = target
            # softmax + CE derivative w.r.t logits = (p - onehot)
            p = F.softmax(self.logits, dim=1)             # [B, C]
            onehot = F.one_hot(targ_idx, num_classes=self.out_dim).float()
            e_y = (onehot - p)                            # sign matches repo's -dCE/dlogits
            gprime = torch.ones_like(p)                   # derivative already accounted in CE/softmax combo
            local_out = e_y                               # [B, C]
        else:
            # BCE derivative wrt pre-activation: (t - y)
            t = target.view(-1)                           # [B]
            y = self.y.view(-1)                           # [B]
            e_y = (t - y).unsqueeze(1)                    # [B, 1]
            gprime = self.dg(self.a_y).unsqueeze(1)       # [B, 1]
            local_out = e_y * gprime                      # [B, 1]

        # ----- dendritic local error via learned feedback theta_y -----
        # e_h = local_out @ theta_y^T  (then × f'(a_h))
        e_h = local_out @ self.theta_y.t()                # [B, M]
        local_h = e_h * self.df(self.a_h)                 # [B, M]

        # ----- parameter updates (three-factor) -----
        # W update
        dW = local_h.t() @ self.x                         # [M, in_dim]
        dW = torch.clamp(dW, -self.clamp, self.clamp)
        self.W.add_(self.lr * dW)

        # soma weights update
        if self.multi_class:
            dV = local_out.t() @ self.h                   # [C, M]
            dV = torch.clamp(dV, -self.clamp, self.clamp)
            self.V.add_(self.lr * dV)
        else:
            dv = (local_out.squeeze(1)[:, None] * self.h).sum(dim=0)  # [M]
            dv = torch.clamp(dv, -self.clamp, self.clamp)
            self.v.add_(self.lr * dv)

        # ----- DLL feedback update (theta_y) -----
        # d theta_y^T  ~  - e_h^T @ local_out   =>  d theta_y ~ - local_out^T @ e_h
        dtheta_y = -(local_out.t() @ e_h) / B             # [out_dim, M] -> average
        dtheta_y = torch.clamp(dtheta_y, -self.clamp, self.clamp).t() # [M, out_dim]
        self.theta_y.add_(self.theta_lr * dtheta_y)

def make_xor(batch_repeats=64, device="cpu"):
    # 4 patterns repeated to make minibatches
    X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], device=device)
    y = torch.tensor([[0.],[1.],[1.],[0.]], device=device)
    X = X.repeat(batch_repeats, 1)
    y = y.repeat(batch_repeats, 1)
    return X, y

def make_parity(n_bits=3, device="cpu"):
    X = torch.stack([((torch.arange(2**n_bits) >> i) & 1).float()
                     for i in range(n_bits)], dim=1)  # [2^n, n_bits] in LSB->MSB order
    y = (X.sum(dim=1) % 2).float().unsqueeze(1)       # parity
    return X.to(device), y.to(device)

train model:

In [ ]:
# ----- tiny trainer -----
def train_dll(model, X, y, epochs=500, noise_std=0.0):
    history = []
    for ep in range(epochs):
        idx = torch.randperm(X.size(0))
        Xb = X[idx]
        yb = y[idx]
        if noise_std > 0:
            Xb = Xb + noise_std * torch.randn_like(Xb)
        out = model(Xb)
        model.dll_step(yb)
        # compute accuracy for logging
        if model.multi_class:
            preds = out.argmax(dim=1)
            targ = (yb if yb.dim()==1 else yb.argmax(dim=1))
            acc = (preds == targ).float().mean().item()
        else:
            acc = (((out > 0.5).float() == yb).float().mean().item())
        history.append(acc)
    return history

section 4 - experiments:

4.1: XOR

4.2: 3-Bit Parity

4.3: Ablation: Dendrite Count

4.4: Robustness to Noise

Section 5 - Baselines:

In [ ]:
bp = torch.nn.Sequential(
    torch.nn.Linear(2, 2), torch.nn.Tanh(),
    torch.nn.Linear(2, 1), torch.nn.Sigmoid()
)
opt = torch.optim.SGD(bp.parameters(), lr=0.05)
loss_fn = torch.nn.BCELoss()

X, y = make_xor()
hist_bp = []
for _ in range(800):
    opt.zero_grad()
    yhat = bp(X)
    loss = loss_fn(yhat, y)
    loss.backward()
    opt.step()
    acc = ((yhat>0.5)==y).float().mean().item()
    hist_bp.append(acc)
plt.plot(hist_bp); plt.title("Backprop XOR Accuracy"); plt.show()


section 5 - results visualization:

explore the data and nonlinear transformation outcomes of dendritic learning models to see how the landscape compares to backprop.

In [5]:
f

NameError: name 'f' is not defined

section 6 - discussion / notes: